# Global Nowcast — REFC correction training (Kaggle / Colab free GPU)

Trains the residual UNet that corrects GFS REFC toward observed precipitation, then
exports `refc_correction.onnx`. Runs in a few hours on a free T4/P100.

**Before running:** enable a GPU accelerator. For the global IMERG target, add your
NASA Earthdata token as a secret named `EARTHDATA_TOKEN` (Kaggle: Add-ons → Secrets;
Colab: set the env var). MRMS (CONUS) needs no credentials.

In [ ]:
# 1. Get the code + training deps
!git clone --depth 1 https://github.com/andrewnakas/globalnowcast.git
%cd globalnowcast
!pip install -q -r requirements-ml.txt onnxruntime

In [ ]:
# 2. Earthdata token (skip if training on MRMS only)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['EARTHDATA_TOKEN'] = UserSecretsClient().get_secret('EARTHDATA_TOKEN')
    print('Earthdata token loaded')
except Exception as e:
    print('No Kaggle secret; set EARTHDATA_TOKEN manually for IMERG:', e)

In [ ]:
# 3. Build a dataset. Start small to validate, then scale up days/every.
#    IMERG = global (needs token). MRMS = CONUS (no token). Build one or both.
!python ml/data/build_dataset.py --source imerg --start 2023-06-01 --days 120 --every 6 --out ml/shards
# !python ml/data/build_dataset.py --source mrms  --start 2023-06-01 --days 120 --every 6 --out ml/shards

In [ ]:
# 4. Train + export ONNX
!python ml/train.py --shards ml/shards --epochs 20 --out ml/model/refc_correction.onnx

In [ ]:
# 5. Sanity-check the exported model on a live GFS frame
import sys; sys.path.insert(0, 'pipeline')
from correct import correct, is_active
from gfs import fetch_refc, find_latest_cycle
from render import decode_refc
from datetime import datetime, timezone
import requests, numpy as np
s = requests.Session()
cyc = find_latest_cycle(s, datetime.now(timezone.utc), 48)
d = decode_refc(fetch_refc(s, cyc, 0))
print('correction active:', is_active())
c = correct(d)
print('mean abs change where echo:', float(np.abs(c - d)[d > -30].mean()))

## 6. Ship it
Download `ml/model/refc_correction.onnx` from the output, commit it to the repo
(it's whitelisted in `.gitignore`), and push. The next hourly GitHub Actions run
applies it automatically and sets `"corrected": true` in `manifest.json`.